In [1]:
!uv pip install pandas numpy xgboost matplotlib seaborn

Resolved 16 packages in 23ms
Installed 16 packages in 1.02s
 + contourpy==1.3.3
 + cycler==0.12.1
 + fonttools==4.63.0
 + kiwisolver==1.5.0
 + matplotlib==3.10.9
 + numpy==2.4.6
 + packaging==26.2
 + pandas==3.0.3
 + pillow==12.2.0
 + pyparsing==3.3.2
 + python-dateutil==2.9.0.post0
 + scipy==1.17.1
 + seaborn==0.13.2
 + six==1.17.0
 + tzdata==2026.2
 + xgboost==3.2.0


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, warnings
warnings.filterwarnings('ignore')

asset_dir = r'c:\Users\loq\Desktop\Trading\finalgo\finalgo-memory-layer\finalgo\08. Model Analysis\30-Minute Vanguard Model\assets'
os.makedirs(asset_dir, exist_ok=True)
fee = 0.0010  # 10 bps strict friction

# Load metadata for features and training split info
with open(r'c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\metadata.json', 'r') as f:
    metadata = json.load(f)
features = metadata['features']

# Load models
long_model = xgb.Booster()
long_model.load_model(r'c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\xgb_long_model.json')
short_model = xgb.Booster()
short_model.load_model(r'c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\xgb_short_model.json')

# Load data
print('Loading 30-min dataset...')
df = pd.read_csv(r'c:\Users\loq\Desktop\Trading\finalgo\data\ranking_data_upstox_30min_1y.csv')
df['Datetime_parsed'] = pd.to_datetime(df['DateTime'])
df = df.dropna(subset=features + ['Next_30Min_Return'])
print(f'Total rows after NA drop: {len(df):,}')

# === OOS SPLIT VERIFICATION ===
# From metadata: train months = 2025-05 to 2026-04, val months = 2026-05
train_months = metadata['production_training']['train_months']
val_months = metadata['production_training']['val_months']
print(f'\nTraining months: {train_months[0]} to {train_months[-1]}')
print(f'Validation months: {val_months}')

# Create the OOS split exactly matching training
df['YearMonth'] = df['Datetime_parsed'].dt.strftime('%Y-%m')
train_df = df[df['YearMonth'].isin(train_months)]
test_df = df[df['YearMonth'].isin(val_months)]

print(f'\n=== TEMPORAL SPLIT VERIFICATION ===')
print(f'Training Set: {len(train_df):,} rows | {train_df["Datetime_parsed"].min()} to {train_df["Datetime_parsed"].max()}')
print(f'Test Set (OOS): {len(test_df):,} rows | {test_df["Datetime_parsed"].min()} to {test_df["Datetime_parsed"].max()}')

# Check for leakage
overlap = set(train_df['Datetime_parsed']) & set(test_df['Datetime_parsed'])
print(f'Overlapping timestamps: {len(overlap)} (must be 0)')

# Generate predictions on OOS ONLY
print('\nGenerating OOS predictions...')
X_oos = test_df[features]
dmatrix_oos = xgb.DMatrix(X_oos)
test_df = test_df.copy()
test_df['Score_Long'] = long_model.predict(dmatrix_oos)
test_df['Score_Short'] = short_model.predict(dmatrix_oos)
test_df['Hour_Min'] = test_df['Datetime_parsed'].dt.strftime('%H:%M')
test_df['Hour'] = test_df['Datetime_parsed'].dt.hour

print(f'OOS predictions generated: {len(test_df):,} rows')
print(f'Score_Long range: [{test_df["Score_Long"].min():.4f}, {test_df["Score_Long"].max():.4f}]')
print(f'Score_Short range: [{test_df["Score_Short"].min():.4f}, {test_df["Score_Short"].max():.4f}]')
print(f'\nScore_Long percentiles:')
for p in [50, 75, 90, 95, 99]:
    print(f'  P{p}: {test_df["Score_Long"].quantile(p/100):.4f}')
print(f'Score_Short percentiles:')
for p in [50, 75, 90, 95, 99]:
    print(f'  P{p}: {test_df["Score_Short"].quantile(p/100):.4f}')
print(f'\nScore_Long negative percentiles:')
for p in [1, 5, 10, 25]:
    print(f'  P{p}: {test_df["Score_Long"].quantile(p/100):.4f}')
print(f'Score_Short negative percentiles:')
for p in [1, 5, 10, 25]:
    print(f'  P{p}: {test_df["Score_Short"].quantile(p/100):.4f}')

Loading 30-min dataset...


Total rows after NA drop: 541,143

Training months: 2025-05 to 2026-04
Validation months: ['2026-05']



=== TEMPORAL SPLIT VERIFICATION ===
Training Set: 501,073 rows | 2025-06-03 14:15:00+05:30 to 2026-04-30 15:15:00+05:30
Test Set (OOS): 40,070 rows | 2026-05-04 09:15:00+05:30 to 2026-05-27 14:45:00+05:30


Overlapping timestamps: 0 (must be 0)

Generating OOS predictions...


OOS predictions generated: 40,070 rows
Score_Long range: [-0.3048, 0.1341]
Score_Short range: [-0.3080, 0.2536]

Score_Long percentiles:
  P50: -0.0467
  P75: 0.0041
  P90: 0.0295
  P95: 0.0403
  P99: 0.0658
Score_Short percentiles:
  P50: -0.0423
  P75: 0.0033
  P90: 0.0259
  P95: 0.0382
  P99: 0.0684

Score_Long negative percentiles:
  P1: -0.2410
  P5: -0.1840
  P10: -0.1404
  P25: -0.0848
Score_Short negative percentiles:
  P1: -0.2647
  P5: -0.2016
  P10: -0.1192
  P25: -0.0802


In [3]:
# ====== PHASE 2: SIGNAL INVERSION DISCOVERY ======
print('='*60)
print('PHASE 2: SIGNAL INVERSION DISCOVERY')
print('='*60)

# 2A. Inverting Short Model -> Long Signal
# If Short Model gives extremely NEGATIVE score (hates the stock for shorts),
# does that mean the stock will go UP?
print('\n--- 2A: Inverting Short Model (Extreme Negative -> Long Signal) ---')
for thresh in [-0.10, -0.12, -0.14, -0.16, -0.18, -0.20, -0.22, -0.24, -0.26]:
    subset = test_df[test_df['Score_Short'] < thresh]
    if len(subset) >= 30:
        wins = (subset['Next_30Min_Return'] > fee).sum()
        wr = wins / len(subset)
        avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000  # bps
        print(f'  Score_Short < {thresh:.2f}: {len(subset):>5} trades | WR: {wr:.1%} | Avg PnL: {avg_pnl:+.1f} bps')

# 2B. Inverting Long Model -> Short Signal  
# If Long Model gives extremely NEGATIVE score (hates the stock for longs),
# does that mean the stock will DROP?
print('\n--- 2B: Inverting Long Model (Extreme Negative -> Short Signal) ---')
for thresh in [-0.10, -0.12, -0.14, -0.16, -0.18, -0.20, -0.22, -0.24]:
    subset = test_df[test_df['Score_Long'] < thresh]
    if len(subset) >= 30:
        wins = (subset['Next_30Min_Return'] < -fee).sum()
        wr = wins / len(subset)
        avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000  # bps
        print(f'  Score_Long < {thresh:.2f}: {len(subset):>5} trades | WR: {wr:.1%} | Avg PnL: {avg_pnl:+.1f} bps')

PHASE 2: SIGNAL INVERSION DISCOVERY

--- 2A: Inverting Short Model (Extreme Negative -> Long Signal) ---
  Score_Short < -0.10:  5594 trades | WR: 39.4% | Avg PnL: -8.2 bps
  Score_Short < -0.12:  3972 trades | WR: 40.8% | Avg PnL: -7.5 bps
  Score_Short < -0.14:  3490 trades | WR: 41.3% | Avg PnL: -7.3 bps
  Score_Short < -0.16:  3197 trades | WR: 41.3% | Avg PnL: -7.9 bps
  Score_Short < -0.18:  2736 trades | WR: 42.0% | Avg PnL: -6.7 bps
  Score_Short < -0.20:  2058 trades | WR: 43.2% | Avg PnL: -5.3 bps
  Score_Short < -0.22:  1452 trades | WR: 45.6% | Avg PnL: -3.4 bps
  Score_Short < -0.24:   908 trades | WR: 46.9% | Avg PnL: -2.1 bps
  Score_Short < -0.26:   471 trades | WR: 51.0% | Avg PnL: +2.1 bps

--- 2B: Inverting Long Model (Extreme Negative -> Short Signal) ---
  Score_Long < -0.10:  7582 trades | WR: 41.4% | Avg PnL: -5.7 bps
  Score_Long < -0.12:  5325 trades | WR: 42.1% | Avg PnL: -4.5 bps
  Score_Long < -0.14:  4023 trades | WR: 43.1% | Avg PnL: -2.9 bps
  Score_Long 

In [4]:
# ====== PHASE 3: SINGLE-MODEL THRESHOLD SWEEPS ======
print('='*60)
print('PHASE 3: SINGLE-MODEL THRESHOLD SWEEPS')
print('='*60)

# 3A: Long Model Only -> Go Long
print('\n--- 3A: Long Model Only (Score_Long > X) -> Go LONG ---')
print(f'{"Threshold":>12} {"Trades":>8} {"WR":>8} {"Avg Profit":>12} {"Cum PnL":>10}')
for thresh in np.arange(0.020, 0.14, 0.005):
    subset = test_df[test_df['Score_Long'] > thresh]
    if len(subset) >= 30:
        wins = (subset['Next_30Min_Return'] > fee).sum()
        wr = wins / len(subset)
        avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000
        cum_pnl = (subset['Next_30Min_Return'] - fee).sum() * 100  # as %
        print(f'  > {thresh:.3f}  {len(subset):>6} {wr:>7.1%} {avg_pnl:>+10.1f} bps {cum_pnl:>+8.1f}%')

# 3B: Short Model Only -> Go Short  
print('\n--- 3B: Short Model Only (Score_Short > X) -> Go SHORT ---')
print(f'{"Threshold":>12} {"Trades":>8} {"WR":>8} {"Avg Profit":>12} {"Cum PnL":>10}')
for thresh in np.arange(0.020, 0.14, 0.005):
    subset = test_df[test_df['Score_Short'] > thresh]
    if len(subset) >= 30:
        wins = (subset['Next_30Min_Return'] < -fee).sum()
        wr = wins / len(subset)
        avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
        cum_pnl = (-subset['Next_30Min_Return'] - fee).sum() * 100
        print(f'  > {thresh:.3f}  {len(subset):>6} {wr:>7.1%} {avg_pnl:>+10.1f} bps {cum_pnl:>+8.1f}%')

PHASE 3: SINGLE-MODEL THRESHOLD SWEEPS

--- 3A: Long Model Only (Score_Long > X) -> Go LONG ---
   Threshold   Trades       WR   Avg Profit    Cum PnL
  > 0.020    6430   39.1%       -6.9 bps   -444.3%
  > 0.025    5163   39.6%       -6.8 bps   -348.9%
  > 0.030    3890   40.3%       -5.8 bps   -224.1%
  > 0.035    2836   41.3%       -4.3 bps   -122.4%
  > 0.040    2051   42.3%       -3.1 bps    -63.0%
  > 0.045    1457   43.2%       -1.8 bps    -26.9%
  > 0.050    1053   44.0%       -1.5 bps    -15.7%
  > 0.055     758   43.9%       -1.4 bps    -10.6%
  > 0.060     565   46.4%       +2.5 bps    +13.9%
  > 0.065     422   48.3%       +4.0 bps    +17.1%
  > 0.070     302   50.3%       +7.5 bps    +22.6%
  > 0.075     216   52.8%      +15.7 bps    +33.8%
  > 0.080     146   56.2%      +21.1 bps    +30.9%
  > 0.085      94   57.4%      +29.4 bps    +27.6%
  > 0.090      45   62.2%      +31.7 bps    +14.3%

--- 3B: Short Model Only (Score_Short > X) -> Go SHORT ---
   Threshold   Trades   

In [5]:
# ====== PHASE 3C: TIME-FILTERED SINGLE-MODEL SWEEPS ======
# The 1-hour model found edge concentrated at 2PM. Let's check all time slots.
print('='*60)
print('PHASE 3C: TIME-FILTERED ANALYSIS')
print('='*60)

# Long Model: Which time slots does the edge appear?
print('\n--- Long Model (Score_Long > 0.070) by Time Slot ---')
subset = test_df[test_df['Score_Long'] > 0.070]
for time_slot in sorted(subset['Hour_Min'].unique()):
    ts = subset[subset['Hour_Min'] == time_slot]
    if len(ts) >= 5:
        wins = (ts['Next_30Min_Return'] > fee).sum()
        wr = wins / len(ts)
        avg_pnl = (ts['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  {time_slot}: {len(ts):>4} trades | WR: {wr:.1%} | Avg PnL: {avg_pnl:+.1f} bps')

# Short Model: Since it globally underperforms, let's see if it has a time pocket
print('\n--- Short Model (Score_Short > 0.050) by Time Slot ---')
subset = test_df[test_df['Score_Short'] > 0.050]
for time_slot in sorted(subset['Hour_Min'].unique()):
    ts = subset[subset['Hour_Min'] == time_slot]
    if len(ts) >= 5:
        wins = (ts['Next_30Min_Return'] < -fee).sum()
        wr = wins / len(ts)
        avg_pnl = (-ts['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  {time_slot}: {len(ts):>4} trades | WR: {wr:.1%} | Avg PnL: {avg_pnl:+.1f} bps')

# Let's try the 2PM window specifically for short model with tighter thresholds
print('\n--- Short Model at 14:15/14:45 ONLY (Higher Thresholds) ---')
pm_df = test_df[test_df['Hour_Min'].isin(['14:15', '14:45'])]
for thresh in np.arange(0.020, 0.12, 0.005):
    subset = pm_df[pm_df['Score_Short'] > thresh]
    if len(subset) >= 20:
        wins = (subset['Next_30Min_Return'] < -fee).sum()
        wr = wins / len(subset)
        avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  > {thresh:.3f}: {len(subset):>4} trades | WR: {wr:.1%} | Avg PnL: {avg_pnl:+.1f} bps')

print('\n--- Long Model at 14:15/14:45 ONLY (Higher Thresholds) ---')
for thresh in np.arange(0.020, 0.10, 0.005):
    subset = pm_df[pm_df['Score_Long'] > thresh]
    if len(subset) >= 20:
        wins = (subset['Next_30Min_Return'] > fee).sum()
        wr = wins / len(subset)
        avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  > {thresh:.3f}: {len(subset):>4} trades | WR: {wr:.1%} | Avg PnL: {avg_pnl:+.1f} bps')

# Also check 15:15 (last slot before close)
print('\n--- Short Model at 15:15 ONLY ---')
eod_df = test_df[test_df['Hour_Min'] == '15:15']
for thresh in np.arange(0.020, 0.10, 0.005):
    subset = eod_df[eod_df['Score_Short'] > thresh]
    if len(subset) >= 10:
        wins = (subset['Next_30Min_Return'] < -fee).sum()
        wr = wins / len(subset)
        avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  > {thresh:.3f}: {len(subset):>4} trades | WR: {wr:.1%} | Avg PnL: {avg_pnl:+.1f} bps')

print('\n--- Long Model at 15:15 ONLY ---')
for thresh in np.arange(0.020, 0.10, 0.005):
    subset = eod_df[eod_df['Score_Long'] > thresh]
    if len(subset) >= 10:
        wins = (subset['Next_30Min_Return'] > fee).sum()
        wr = wins / len(subset)
        avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  > {thresh:.3f}: {len(subset):>4} trades | WR: {wr:.1%} | Avg PnL: {avg_pnl:+.1f} bps')

PHASE 3C: TIME-FILTERED ANALYSIS

--- Long Model (Score_Long > 0.070) by Time Slot ---
  09:45:    5 trades | WR: 20.0% | Avg PnL: -9.7 bps
  10:15:    6 trades | WR: 50.0% | Avg PnL: -14.4 bps
  10:45:    5 trades | WR: 80.0% | Avg PnL: +56.0 bps
  12:15:    8 trades | WR: 37.5% | Avg PnL: -11.7 bps
  13:15:    8 trades | WR: 37.5% | Avg PnL: -16.8 bps
  13:45:   20 trades | WR: 45.0% | Avg PnL: +3.0 bps
  14:15:   13 trades | WR: 38.5% | Avg PnL: -10.2 bps
  14:45:   60 trades | WR: 46.7% | Avg PnL: -7.2 bps
  15:15:  167 trades | WR: 54.5% | Avg PnL: +16.8 bps

--- Short Model (Score_Short > 0.050) by Time Slot ---
  09:15:   57 trades | WR: 35.1% | Avg PnL: -25.8 bps
  09:45:   44 trades | WR: 40.9% | Avg PnL: -17.7 bps
  10:15:   35 trades | WR: 42.9% | Avg PnL: -22.5 bps
  10:45:   25 trades | WR: 44.0% | Avg PnL: -6.3 bps
  11:15:   34 trades | WR: 44.1% | Avg PnL: -26.6 bps
  11:45:   28 trades | WR: 46.4% | Avg PnL: -10.4 bps
  12:15:   29 trades | WR: 44.8% | Avg PnL: -19.8 b

In [6]:
# ====== PHASE 4: DUAL-LOCK SWEEPS & DEAD END TESTS ======
print('='*60)
print('PHASE 4: DUAL-LOCK SWEEPS & DEAD END TESTS')
print('='*60)

# 4A: Agreement Long (Both Models Say BUY)
print('\n--- 4A: DEAD END TEST: Agreement Long (Both Models Positive) ---')
found = False
for lt in np.arange(0.02, 0.10, 0.01):
    for st in np.arange(0.02, 0.10, 0.01):
        subset = test_df[(test_df['Score_Long'] > lt) & (test_df['Score_Short'] > st)]
        if len(subset) >= 30:
            wins = (subset['Next_30Min_Return'] > fee).sum()
            wr = wins / len(subset)
            avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000
            if wr > 0.50:
                print(f'  L>{lt:.2f} S>{st:.2f}: {len(subset):>4} trades | WR: {wr:.1%} | PnL: {avg_pnl:+.1f} bps')
                found = True
if not found:
    print('  ZERO valid configurations with WR > 50%. CONFIRMED DEAD END.')

# 4B: Agreement Short (Both Models Say SELL) 
print('\n--- 4B: DEAD END TEST: Agreement Short (Both Models Negative) ---')
found = False
for lt in np.arange(-0.08, -0.26, -0.02):
    for st in np.arange(-0.08, -0.26, -0.02):
        subset = test_df[(test_df['Score_Long'] < lt) & (test_df['Score_Short'] < st)]
        if len(subset) >= 30:
            wins = (subset['Next_30Min_Return'] < -fee).sum()
            wr = wins / len(subset)
            avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
            if wr > 0.50:
                print(f'  L<{lt:.2f} S<{st:.2f}: {len(subset):>4} trades | WR: {wr:.1%} | PnL: {avg_pnl:+.1f} bps')
                found = True
if not found:
    print('  ZERO valid configurations with WR > 50%. CONFIRMED DEAD END.')

# 4C: Score Spread  
print('\n--- 4C: DEAD END TEST: Score Spread (Long - Short) ---')
test_df['Spread'] = test_df['Score_Long'] - test_df['Score_Short']
spread_bins = pd.qcut(test_df['Spread'], 20, labels=False, duplicates='drop')
for b in sorted(spread_bins.unique()):
    subset = test_df[spread_bins == b]
    long_wr = (subset['Next_30Min_Return'] > fee).mean()
    short_wr = (subset['Next_30Min_Return'] < -fee).mean()
    if long_wr > 0.52 or short_wr > 0.52:
        print(f'  Bin {b:>2}: {len(subset):>5} trades | Long WR: {long_wr:.1%} | Short WR: {short_wr:.1%}')
print('  (Only bins with WR > 52% shown. If none, spread is dead.)')

# 4D: Score Ratio
print('\n--- 4D: DEAD END TEST: Score Ratio (Long / Short) ---')
# Avoid division by zero
mask = test_df['Score_Short'].abs() > 0.001
temp = test_df[mask].copy()
temp['Ratio'] = temp['Score_Long'] / temp['Score_Short']
ratio_bins = pd.qcut(temp['Ratio'], 20, labels=False, duplicates='drop')
for b in sorted(ratio_bins.unique()):
    subset = temp[ratio_bins == b]
    long_wr = (subset['Next_30Min_Return'] > fee).mean()
    short_wr = (subset['Next_30Min_Return'] < -fee).mean()
    if long_wr > 0.52 or short_wr > 0.52:
        print(f'  Bin {b:>2}: {len(subset):>5} trades | Long WR: {long_wr:.1%} | Short WR: {short_wr:.1%}')
print('  (Only bins with WR > 52% shown. If none, ratio is dead.)')

PHASE 4: DUAL-LOCK SWEEPS & DEAD END TESTS

--- 4A: DEAD END TEST: Agreement Long (Both Models Positive) ---
  ZERO valid configurations with WR > 50%. CONFIRMED DEAD END.

--- 4B: DEAD END TEST: Agreement Short (Both Models Negative) ---
  ZERO valid configurations with WR > 50%. CONFIRMED DEAD END.

--- 4C: DEAD END TEST: Score Spread (Long - Short) ---
  (Only bins with WR > 52% shown. If none, spread is dead.)

--- 4D: DEAD END TEST: Score Ratio (Long / Short) ---


  (Only bins with WR > 52% shown. If none, ratio is dead.)


In [7]:
# ====== PHASE 4E: DUAL-LOCK LONG (L>X AND S<Y) ======
print('='*60)
print('PHASE 4E: DUAL-LOCK LONG (Score_Long > X AND Score_Short < Y)')
print('='*60)

results_dl = []
for lt in np.arange(0.040, 0.10, 0.005):
    for st in np.arange(-0.04, -0.28, -0.02):
        subset = test_df[(test_df['Score_Long'] > lt) & (test_df['Score_Short'] < st)]
        if len(subset) >= 30:
            wins = (subset['Next_30Min_Return'] > fee).sum()
            wr = wins / len(subset)
            avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000
            cum_pnl = (subset['Next_30Min_Return'] - fee).sum() * 100
            results_dl.append({'L_thresh': lt, 'S_thresh': st, 'Trades': len(subset), 
                              'WR': wr, 'Avg_PnL_bps': avg_pnl, 'Cum_PnL_pct': cum_pnl})

results_dl_df = pd.DataFrame(results_dl)
if len(results_dl_df) > 0:
    # Show top 15 by Win Rate
    top = results_dl_df.sort_values('WR', ascending=False).head(15)
    print('\nTop 15 Dual-Lock Long configs by Win Rate:')
    for _, r in top.iterrows():
        print(f'  L>{r["L_thresh"]:.3f} S<{r["S_thresh"]:.2f}: {r["Trades"]:>4} trades | WR: {r["WR"]:.1%} | PnL: {r["Avg_PnL_bps"]:+.1f} bps | Cum: {r["Cum_PnL_pct"]:+.1f}%')
else:
    print('No valid Dual-Lock Long configurations found.')

# ====== PHASE 4F: DUAL-LOCK SHORT (S>X AND L<Y) ======
print('\n' + '='*60)
print('PHASE 4F: DUAL-LOCK SHORT (Score_Short > X AND Score_Long < Y)')
print('='*60)

results_ds = []
for st in np.arange(0.020, 0.10, 0.005):
    for lt in np.arange(-0.04, -0.28, -0.02):
        subset = test_df[(test_df['Score_Short'] > st) & (test_df['Score_Long'] < lt)]
        if len(subset) >= 30:
            wins = (subset['Next_30Min_Return'] < -fee).sum()
            wr = wins / len(subset)
            avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
            cum_pnl = (-subset['Next_30Min_Return'] - fee).sum() * 100
            results_ds.append({'S_thresh': st, 'L_thresh': lt, 'Trades': len(subset),
                              'WR': wr, 'Avg_PnL_bps': avg_pnl, 'Cum_PnL_pct': cum_pnl})

results_ds_df = pd.DataFrame(results_ds)
if len(results_ds_df) > 0:
    top = results_ds_df.sort_values('WR', ascending=False).head(15)
    print('\nTop 15 Dual-Lock Short configs by Win Rate:')
    for _, r in top.iterrows():
        print(f'  S>{r["S_thresh"]:.3f} L<{r["L_thresh"]:.2f}: {r["Trades"]:>4} trades | WR: {r["WR"]:.1%} | PnL: {r["Avg_PnL_bps"]:+.1f} bps | Cum: {r["Cum_PnL_pct"]:+.1f}%')
else:
    print('No valid Dual-Lock Short configurations found.')

PHASE 4E: DUAL-LOCK LONG (Score_Long > X AND Score_Short < Y)

Top 15 Dual-Lock Long configs by Win Rate:
  L>0.090 S<-0.06: 45.0 trades | WR: 62.2% | PnL: +31.7 bps | Cum: +14.3%
  L>0.090 S<-0.04: 45.0 trades | WR: 62.2% | PnL: +31.7 bps | Cum: +14.3%
  L>0.090 S<-0.10: 44.0 trades | WR: 61.4% | PnL: +28.4 bps | Cum: +12.5%
  L>0.090 S<-0.08: 44.0 trades | WR: 61.4% | PnL: +28.4 bps | Cum: +12.5%
  L>0.090 S<-0.24: 41.0 trades | WR: 61.0% | PnL: +23.9 bps | Cum: +9.8%
  L>0.090 S<-0.18: 43.0 trades | WR: 60.5% | PnL: +28.1 bps | Cum: +12.1%
  L>0.090 S<-0.12: 43.0 trades | WR: 60.5% | PnL: +28.1 bps | Cum: +12.1%
  L>0.090 S<-0.16: 43.0 trades | WR: 60.5% | PnL: +28.1 bps | Cum: +12.1%
  L>0.090 S<-0.20: 43.0 trades | WR: 60.5% | PnL: +28.1 bps | Cum: +12.1%
  L>0.090 S<-0.14: 43.0 trades | WR: 60.5% | PnL: +28.1 bps | Cum: +12.1%
  L>0.090 S<-0.22: 42.0 trades | WR: 59.5% | PnL: +19.2 bps | Cum: +8.1%
  L>0.085 S<-0.20: 83.0 trades | WR: 57.8% | PnL: +31.8 bps | Cum: +26.4%
  L>0.08


Top 15 Dual-Lock Short configs by Win Rate:
  S>0.090 L<-0.26: 61.0 trades | WR: 54.1% | PnL: -0.9 bps | Cum: -0.6%
  S>0.085 L<-0.26: 73.0 trades | WR: 53.4% | PnL: +0.2 bps | Cum: +0.2%
  S>0.070 L<-0.26: 106.0 trades | WR: 52.8% | PnL: +7.2 bps | Cum: +7.6%
  S>0.090 L<-0.24: 82.0 trades | WR: 52.4% | PnL: -2.3 bps | Cum: -1.9%
  S>0.090 L<-0.22: 90.0 trades | WR: 52.2% | PnL: -3.8 bps | Cum: -3.5%
  S>0.060 L<-0.26: 135.0 trades | WR: 51.9% | PnL: +8.7 bps | Cum: +11.7%
  S>0.075 L<-0.26: 93.0 trades | WR: 51.6% | PnL: +3.5 bps | Cum: +3.2%
  S>0.085 L<-0.14: 144.0 trades | WR: 51.4% | PnL: +5.2 bps | Cum: +7.5%
  S>0.045 L<-0.26: 154.0 trades | WR: 51.3% | PnL: +7.6 bps | Cum: +11.8%
  S>0.040 L<-0.26: 156.0 trades | WR: 51.3% | PnL: +7.5 bps | Cum: +11.7%
  S>0.090 L<-0.14: 117.0 trades | WR: 51.3% | PnL: +4.1 bps | Cum: +4.8%
  S>0.020 L<-0.26: 162.0 trades | WR: 51.2% | PnL: +7.3 bps | Cum: +11.8%
  S>0.085 L<-0.24: 100.0 trades | WR: 51.0% | PnL: -3.4 bps | Cum: -3.4%
  S>0.0

In [8]:
# ====== PHASE 4G: DUAL-LOCK AT SPECIFIC TIME SLOTS ======
print('='*60)
print('PHASE 4G: DUAL-LOCK AT 15:15 (EOD) ONLY')
print('='*60)

eod = test_df[test_df['Hour_Min'] == '15:15']
print(f'Total EOD (15:15) rows: {len(eod):,}')

# Dual-Lock Long at 15:15
print('\n--- Dual-Lock Long at 15:15 ---')
results = []
for lt in np.arange(0.030, 0.10, 0.005):
    for st in np.arange(-0.04, -0.28, -0.02):
        subset = eod[(eod['Score_Long'] > lt) & (eod['Score_Short'] < st)]
        if len(subset) >= 15:
            wins = (subset['Next_30Min_Return'] > fee).sum()
            wr = wins / len(subset)
            avg_pnl = (subset['Next_30Min_Return'] - fee).mean() * 10000
            results.append({'L': lt, 'S': st, 'N': len(subset), 'WR': wr, 'PnL': avg_pnl})
if results:
    rdf = pd.DataFrame(results).sort_values('WR', ascending=False).head(20)
    for _, r in rdf.iterrows():
        print(f'  L>{r["L"]:.3f} S<{r["S"]:.2f}: {int(r["N"]):>4} trades | WR: {r["WR"]:.1%} | PnL: {r["PnL"]:+.1f} bps')
else:
    print('  No valid configs.')

# Dual-Lock Short at 15:15
print('\n--- Dual-Lock Short at 15:15 ---')
results = []
for st in np.arange(0.020, 0.10, 0.005):
    for lt in np.arange(-0.04, -0.28, -0.02):
        subset = eod[(eod['Score_Short'] > st) & (eod['Score_Long'] < lt)]
        if len(subset) >= 15:
            wins = (subset['Next_30Min_Return'] < -fee).sum()
            wr = wins / len(subset)
            avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
            results.append({'S': st, 'L': lt, 'N': len(subset), 'WR': wr, 'PnL': avg_pnl})
if results:
    rdf = pd.DataFrame(results).sort_values('WR', ascending=False).head(20)
    for _, r in rdf.iterrows():
        print(f'  S>{r["S"]:.3f} L<{r["L"]:.2f}: {int(r["N"]):>4} trades | WR: {r["WR"]:.1%} | PnL: {r["PnL"]:+.1f} bps')
else:
    print('  No valid configs.')

# Also check Short Model ONLY at 14:15 (where it showed 56.3% WR earlier)
print('\n' + '='*60)
print('PHASE 4H: SHORT MODEL ONLY AT 14:15 (Best Short Time Slot)')
print('='*60)
pm_1415 = test_df[test_df['Hour_Min'] == '14:15']
print(f'Total 14:15 rows: {len(pm_1415):,}')
for thresh in np.arange(0.020, 0.12, 0.005):
    subset = pm_1415[pm_1415['Score_Short'] > thresh]
    if len(subset) >= 10:
        wins = (subset['Next_30Min_Return'] < -fee).sum()
        wr = wins / len(subset)
        avg_pnl = (-subset['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  S>{thresh:.3f}: {len(subset):>4} trades | WR: {wr:.1%} | PnL: {avg_pnl:+.1f} bps')

PHASE 4G: DUAL-LOCK AT 15:15 (EOD) ONLY
Total EOD (15:15) rows: 2,924

--- Dual-Lock Long at 15:15 ---
  L>0.095 S<-0.24:   18 trades | WR: 66.7% | PnL: +35.8 bps
  L>0.095 S<-0.04:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.06:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.20:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.14:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.08:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.18:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.10:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.12:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.16:   20 trades | WR: 65.0% | PnL: +43.7 bps
  L>0.095 S<-0.22:   19 trades | WR: 63.2% | PnL: +24.8 bps
  L>0.090 S<-0.24:   34 trades | WR: 61.8% | PnL: +27.7 bps
  L>0.090 S<-0.12:   36 trades | WR: 61.1% | PnL: +32.6 bps
  L>0.090 S<-0.16:   36 trades | WR: 61.1% | PnL: +32.6 bps
  L>0.090 S<-0.10:   36 trades | WR: 61.1% | PnL: +32.6 b

  S>0.090 L<-0.04:   76 trades | WR: 56.6% | PnL: +10.7 bps
  S>0.090 L<-0.16:   75 trades | WR: 56.0% | PnL: +9.7 bps
  S>0.090 L<-0.08:   75 trades | WR: 56.0% | PnL: +9.7 bps
  S>0.090 L<-0.12:   75 trades | WR: 56.0% | PnL: +9.7 bps
  S>0.090 L<-0.18:   75 trades | WR: 56.0% | PnL: +9.7 bps
  S>0.090 L<-0.14:   75 trades | WR: 56.0% | PnL: +9.7 bps
  S>0.090 L<-0.10:   75 trades | WR: 56.0% | PnL: +9.7 bps
  S>0.090 L<-0.06:   75 trades | WR: 56.0% | PnL: +9.7 bps
  S>0.090 L<-0.20:   73 trades | WR: 54.8% | PnL: +7.6 bps
  S>0.085 L<-0.04:   90 trades | WR: 54.4% | PnL: +8.7 bps
  S>0.090 L<-0.22:   70 trades | WR: 54.3% | PnL: -1.3 bps
  S>0.020 L<-0.22:  410 trades | WR: 54.1% | PnL: +11.5 bps
  S>0.085 L<-0.10:   89 trades | WR: 53.9% | PnL: +7.9 bps
  S>0.085 L<-0.18:   89 trades | WR: 53.9% | PnL: +7.9 bps
  S>0.085 L<-0.08:   89 trades | WR: 53.9% | PnL: +7.9 bps
  S>0.085 L<-0.06:   89 trades | WR: 53.9% | PnL: +7.9 bps
  S>0.085 L<-0.16:   89 trades | WR: 53.9% | PnL: +7.9

In [9]:
# ====== PHASE 5: GENERATE ALL VISUALIZATIONS ======
print('Generating all publication-quality visualizations...')

plt.rcParams.update({'font.size': 10, 'figure.dpi': 150})

# === 5A: Time-of-Day Conviction Heatmap (Long) ===
test_df['Long_Quintile'] = pd.qcut(test_df['Score_Long'], 5, labels=['Q1(Low)','Q2','Q3','Q4','Q5(High)'])
pivot_long = test_df.pivot_table(index='Long_Quintile', columns='Hour_Min', 
                                  values='Next_30Min_Return', aggfunc='mean') * 10000  # bps

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_long, annot=True, cmap='RdYlGn', fmt='.1f', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Avg PnL (bps)'})
ax.set_title('30-Min Long Model: Time of Day vs Score Conviction (Avg PnL in bps, raw before fees)', fontsize=13)
ax.set_xlabel('Time of Day (IST)')
ax.set_ylabel('Long Score Quintile')
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'time_of_day_conviction_long.png'), dpi=200)
plt.close()
print('Saved time_of_day_conviction_long.png')

# === 5B: Time-of-Day Conviction Heatmap (Short) ===
test_df['Short_Quintile'] = pd.qcut(test_df['Score_Short'], 5, labels=['Q1(Low)','Q2','Q3','Q4','Q5(High)'])
pivot_short = test_df.pivot_table(index='Short_Quintile', columns='Hour_Min',
                                   values='Next_30Min_Return', aggfunc='mean') * -10000  # bps (inverted for short)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot_short, annot=True, cmap='RdYlGn', fmt='.1f', center=0, ax=ax,
            linewidths=0.5, cbar_kws={'label': 'Avg Short PnL (bps)'})
ax.set_title('30-Min Short Model: Time of Day vs Score Conviction (Avg Short PnL in bps, raw before fees)', fontsize=13)
ax.set_xlabel('Time of Day (IST)')
ax.set_ylabel('Short Score Quintile')
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'time_of_day_conviction_short.png'), dpi=200)
plt.close()
print('Saved time_of_day_conviction_short.png')

Generating all publication-quality visualizations...


Saved time_of_day_conviction_long.png


Saved time_of_day_conviction_short.png


In [10]:
# === 5C: OOS Calibration Curves ===
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Long Model -> Long trades
thresholds = np.arange(0.020, 0.10, 0.002)
wr_long, trades_long, pnl_long = [], [], []
for t in thresholds:
    s = test_df[test_df['Score_Long'] > t]
    if len(s) >= 30:
        wr_long.append((s['Next_30Min_Return'] > fee).mean())
        trades_long.append(len(s))
        pnl_long.append((s['Next_30Min_Return'] - fee).mean() * 10000)
    else:
        wr_long.append(np.nan); trades_long.append(0); pnl_long.append(np.nan)

ax = axes[0]
ax.plot(thresholds, wr_long, 'g-', linewidth=2)
ax.axhline(0.5, color='black', linestyle='--', alpha=0.7)
ax.set_title('Long Model → Long Trades')
ax.set_xlabel('Score_Long Threshold')
ax.set_ylabel('Win Rate (net 10bps)')
ax.set_ylim(0.35, 0.70)
ax.grid(True, alpha=0.3)

# Short Model -> Short trades
thresholds_s = np.arange(0.020, 0.14, 0.002)
wr_short, trades_short = [], []
for t in thresholds_s:
    s = test_df[test_df['Score_Short'] > t]
    if len(s) >= 30:
        wr_short.append((s['Next_30Min_Return'] < -fee).mean())
        trades_short.append(len(s))
    else:
        wr_short.append(np.nan); trades_short.append(0)

ax = axes[1]
ax.plot(thresholds_s, wr_short, 'r-', linewidth=2)
ax.axhline(0.5, color='black', linestyle='--', alpha=0.7)
ax.set_title('Short Model → Short Trades')
ax.set_xlabel('Score_Short Threshold')
ax.set_ylabel('Win Rate (net 10bps)')
ax.set_ylim(0.35, 0.70)
ax.grid(True, alpha=0.3)

# Long Model at 15:15 ONLY
thresholds_eod = np.arange(0.020, 0.10, 0.002)
wr_eod, trades_eod = [], []
eod_only = test_df[test_df['Hour_Min'] == '15:15']
for t in thresholds_eod:
    s = eod_only[eod_only['Score_Long'] > t]
    if len(s) >= 15:
        wr_eod.append((s['Next_30Min_Return'] > fee).mean())
        trades_eod.append(len(s))
    else:
        wr_eod.append(np.nan); trades_eod.append(0)

ax = axes[2]
ax.plot(thresholds_eod, wr_eod, 'b-', linewidth=2)
ax.axhline(0.5, color='black', linestyle='--', alpha=0.7)
ax.set_title('Long Model at 15:15 ONLY → Long Trades')
ax.set_xlabel('Score_Long Threshold')
ax.set_ylabel('Win Rate (net 10bps)')
ax.set_ylim(0.35, 0.70)
ax.grid(True, alpha=0.3)

plt.suptitle('30-Min Model OOS Calibration Curves (Net 10 bps Friction)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'oos_calibration.png'), dpi=200, bbox_inches='tight')
plt.close()
print('Saved oos_calibration.png')

Saved oos_calibration.png


In [11]:
# === 5D: Dual-Lock Heatmaps ===

# Dual Long Heatmap: WR for (Score_Long > X AND Score_Short < Y) at 15:15
eod = test_df[test_df['Hour_Min'] == '15:15'].copy()

lt_range = np.arange(0.030, 0.10, 0.005)
st_range = np.arange(-0.04, -0.26, -0.02)

heat_long = pd.DataFrame(index=[f'>{t:.3f}' for t in lt_range], 
                          columns=[f'<{t:.2f}' for t in st_range])
for lt in lt_range:
    for st in st_range:
        subset = eod[(eod['Score_Long'] > lt) & (eod['Score_Short'] < st)]
        if len(subset) >= 10:
            wr = (subset['Next_30Min_Return'] > fee).mean()
            heat_long.loc[f'>{lt:.3f}', f'<{st:.2f}'] = wr
        else:
            heat_long.loc[f'>{lt:.3f}', f'<{st:.2f}'] = np.nan

heat_long = heat_long.astype(float)
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(heat_long, annot=True, cmap='RdYlGn', fmt='.2f', center=0.50, ax=ax,
            linewidths=0.5, vmin=0.40, vmax=0.70, cbar_kws={'label': 'Win Rate'})
ax.set_title('Dual-Lock LONG at 15:15: Win Rate Grid (Net 10bps)', fontsize=13)
ax.set_xlabel('Score_Short Filter (<)')
ax.set_ylabel('Score_Long Threshold (>)')
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'dual_long_heatmap.png'), dpi=200)
plt.close()
print('Saved dual_long_heatmap.png')

# Dual Short Heatmap: WR for (Score_Short > X AND Score_Long < Y) at 15:15
st_range2 = np.arange(0.020, 0.10, 0.005)
lt_range2 = np.arange(-0.04, -0.28, -0.02)

heat_short = pd.DataFrame(index=[f'>{t:.3f}' for t in st_range2],
                           columns=[f'<{t:.2f}' for t in lt_range2])
for st in st_range2:
    for lt in lt_range2:
        subset = eod[(eod['Score_Short'] > st) & (eod['Score_Long'] < lt)]
        if len(subset) >= 10:
            wr = (subset['Next_30Min_Return'] < -fee).mean()
            heat_short.loc[f'>{st:.3f}', f'<{lt:.2f}'] = wr
        else:
            heat_short.loc[f'>{st:.3f}', f'<{lt:.2f}'] = np.nan

heat_short = heat_short.astype(float)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(heat_short, annot=True, cmap='RdYlGn', fmt='.2f', center=0.50, ax=ax,
            linewidths=0.5, vmin=0.35, vmax=0.65, cbar_kws={'label': 'Win Rate'})
ax.set_title('Dual-Lock SHORT at 15:15: Win Rate Grid (Net 10bps)', fontsize=13)
ax.set_xlabel('Score_Long Filter (<)')
ax.set_ylabel('Score_Short Threshold (>)')
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'dual_short_heatmap.png'), dpi=200)
plt.close()
print('Saved dual_short_heatmap.png')

Saved dual_long_heatmap.png


Saved dual_short_heatmap.png


In [12]:
# === 5E: Complete Edge Map (Decile x Decile WR for both Long & Short) ===
test_df['Long_Decile'] = pd.qcut(test_df['Score_Long'], 10, labels=False, duplicates='drop')
test_df['Short_Decile'] = pd.qcut(test_df['Score_Short'], 10, labels=False, duplicates='drop')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# Long Win Rate grid
long_wr = test_df.groupby(['Long_Decile', 'Short_Decile']).apply(
    lambda x: (x['Next_30Min_Return'] > fee).mean()
).unstack()
sns.heatmap(long_wr, annot=True, cmap='RdYlGn', fmt='.2f', center=0.50, ax=ax1,
            linewidths=0.5, vmin=0.35, vmax=0.60)
ax1.set_title('Long Win Rate by Score Deciles (Net 10bps)')
ax1.set_xlabel('Short Score Decile')
ax1.set_ylabel('Long Score Decile')
ax1.invert_yaxis()

# Short Win Rate grid
short_wr = test_df.groupby(['Long_Decile', 'Short_Decile']).apply(
    lambda x: (x['Next_30Min_Return'] < -fee).mean()
).unstack()
sns.heatmap(short_wr, annot=True, cmap='RdYlGn', fmt='.2f', center=0.50, ax=ax2,
            linewidths=0.5, vmin=0.35, vmax=0.60)
ax2.set_title('Short Win Rate by Score Deciles (Net 10bps)')
ax2.set_xlabel('Short Score Decile')
ax2.set_ylabel('Long Score Decile')
ax2.invert_yaxis()

plt.suptitle('30-Min Models: Complete Edge Map (All Hours)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'complete_edge_map.png'), dpi=200, bbox_inches='tight')
plt.close()
print('Saved complete_edge_map.png')

Saved complete_edge_map.png


In [13]:
# ====== PHASE 6: QUARTERLY CONSISTENCY ======
print('='*60)
print('PHASE 6: QUARTERLY/WEEKLY CONSISTENCY')
print('='*60)

# OOS is only May 2026, so we'll break it by week
test_df['Week'] = test_df['Datetime_parsed'].dt.isocalendar().week.astype(int)
test_df['WeekStart'] = test_df['Datetime_parsed'].dt.to_period('W').apply(lambda x: x.start_time.strftime('%Y-%m-%d'))

# Long Strategy performance by week (Score_Long > 0.075)
print('\n--- Long Strategy (Score_Long > 0.075) by Week ---')
long_strat = test_df[test_df['Score_Long'] > 0.075]
for week in sorted(long_strat['WeekStart'].unique()):
    w = long_strat[long_strat['WeekStart'] == week]
    if len(w) >= 5:
        wr = (w['Next_30Min_Return'] > fee).mean()
        avg_pnl = (w['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  {week}: {len(w):>3} trades | WR: {wr:.1%} | PnL: {avg_pnl:+.1f} bps')

# Short Strategy at 14:15 by week (Score_Short > 0.050 at 14:15)
print('\n--- Short Strategy (Score_Short > 0.050 at 14:15) by Week ---')
short_strat = test_df[(test_df['Score_Short'] > 0.050) & (test_df['Hour_Min'] == '14:15')]
for week in sorted(short_strat['WeekStart'].unique()):
    w = short_strat[short_strat['WeekStart'] == week]
    if len(w) >= 5:
        wr = (w['Next_30Min_Return'] < -fee).mean()
        avg_pnl = (-w['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  {week}: {len(w):>3} trades | WR: {wr:.1%} | PnL: {avg_pnl:+.1f} bps')

# Long at 15:15 by week
print('\n--- Long Strategy at 15:15 (Score_Long > 0.075) by Week ---')
long_eod = test_df[(test_df['Score_Long'] > 0.075) & (test_df['Hour_Min'] == '15:15')]
for week in sorted(long_eod['WeekStart'].unique()):
    w = long_eod[long_eod['WeekStart'] == week]
    if len(w) >= 3:
        wr = (w['Next_30Min_Return'] > fee).mean()
        avg_pnl = (w['Next_30Min_Return'] - fee).mean() * 10000
        print(f'  {week}: {len(w):>3} trades | WR: {wr:.1%} | PnL: {avg_pnl:+.1f} bps')

# Generate weekly consistency chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Long all-hours
long_weekly = test_df[test_df['Score_Long'] > 0.075].groupby('WeekStart').apply(
    lambda x: pd.Series({
        'WR': (x['Next_30Min_Return'] > fee).mean(),
        'PnL': (x['Next_30Min_Return'] - fee).mean() * 10000,
        'N': len(x)
    })
)
if len(long_weekly) > 0:
    ax = axes[0]
    ax.bar(range(len(long_weekly)), long_weekly['WR'], color=['green' if w > 0.5 else 'red' for w in long_weekly['WR']])
    ax.axhline(0.5, color='black', linestyle='--')
    ax.set_xticks(range(len(long_weekly)))
    ax.set_xticklabels(long_weekly.index, rotation=45, ha='right', fontsize=8)
    ax.set_title('Long Model (>0.075) Weekly Win Rate')
    ax.set_ylabel('Win Rate')

# Long at 15:15
long_eod_weekly = test_df[(test_df['Score_Long'] > 0.060) & (test_df['Hour_Min'] == '15:15')].groupby('WeekStart').apply(
    lambda x: pd.Series({
        'WR': (x['Next_30Min_Return'] > fee).mean(),
        'PnL': (x['Next_30Min_Return'] - fee).mean() * 10000,
        'N': len(x)
    })
)
if len(long_eod_weekly) > 0:
    ax = axes[1]
    ax.bar(range(len(long_eod_weekly)), long_eod_weekly['WR'], 
           color=['green' if w > 0.5 else 'red' for w in long_eod_weekly['WR']])
    ax.axhline(0.5, color='black', linestyle='--')
    ax.set_xticks(range(len(long_eod_weekly)))
    ax.set_xticklabels(long_eod_weekly.index, rotation=45, ha='right', fontsize=8)
    ax.set_title('Long Model at 15:15 (>0.060) Weekly Win Rate')
    ax.set_ylabel('Win Rate')

plt.suptitle('30-Min Model: Weekly Consistency (OOS May 2026)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(asset_dir, 'weekly_consistency.png'), dpi=200, bbox_inches='tight')
plt.close()
print('Saved weekly_consistency.png')

PHASE 6: QUARTERLY/WEEKLY CONSISTENCY



--- Long Strategy (Score_Long > 0.075) by Week ---
  2026-05-04:  55 trades | WR: 61.8% | PnL: +31.4 bps
  2026-05-11:  50 trades | WR: 28.0% | PnL: -34.5 bps
  2026-05-18:  79 trades | WR: 60.8% | PnL: +33.0 bps
  2026-05-25:  32 trades | WR: 56.2% | PnL: +24.2 bps

--- Short Strategy (Score_Short > 0.050 at 14:15) by Week ---
  2026-05-04:  53 trades | WR: 60.4% | PnL: +6.3 bps
  2026-05-11:  55 trades | WR: 65.5% | PnL: +18.9 bps
  2026-05-18:  55 trades | WR: 54.5% | PnL: +4.7 bps
  2026-05-25:  27 trades | WR: 33.3% | PnL: -35.0 bps

--- Long Strategy at 15:15 (Score_Long > 0.075) by Week ---
  2026-05-04:  42 trades | WR: 54.8% | PnL: +37.6 bps
  2026-05-11:  30 trades | WR: 26.7% | PnL: -59.2 bps
  2026-05-18:  42 trades | WR: 78.6% | PnL: +69.2 bps
  2026-05-25:  21 trades | WR: 66.7% | PnL: +42.8 bps


Saved weekly_consistency.png
